# ConFit v2 on Djinni with LLM-panel labels

Trains the **official ConFit v2 code** (github.com/jasonyux/ConFit-v2, commit `8022a9e`, MIT) on the
panel-labelled Djinni pairs, with both of its parts:
1. **HyRe**: an LLM writes a hypothetical ideal resume for every job, appended to the job text (the paper used
   GPT-4o-mini; here Qwen2.5-7B-Instruct, the open model named in Workbook 1).
2. **RUHM**: train once, mine runner-up hard negatives with that model, then train again with them.

Encoder: `jinaai/jina-embeddings-v2-base-en` (the paper's Jina v2 encoder, English version).
Five small compatibility patches are applied to the repo (listed in the patch cell); none changes the method.

**Run:**
1. Upload `pairs.zip` (the `pairs` folder after `aggregate.py --apply`) to **My Drive/confit_run/**.
   If it is not there, the data cell asks you to upload it instead.
2. Runtime → Change runtime type → **A100 GPU**.
3. Run all once with `SMOKE = True` (quick trial: 1 epoch, no RUHM; about 20 to 30 minutes).
4. Set `SMOKE = False` and Run all again for the real run (about 1 to 2 hours).

HyRe outputs, each stage's checkpoints and the results are saved to My Drive/confit_run/smoke or
My Drive/confit_run/full, and a rerun resumes finished stages from there. At the end `confit_results.zip`
downloads; it contains `confit_scores.csv` for `score_on_panel.py`.

In [ ]:
!git clone -q https://github.com/jasonyux/ConFit-v2.git /content/ConFit-v2
!cd /content/ConFit-v2 && git checkout -q 8022a9ee372664b35acad313abb4422de6d123f0 && git log -1 --format='%H %cd'
!pip -q install "transformers<5" pytorch_lightning torchmetrics wandb faiss-cpu jieba nltk debugpy xgboost deepspeed accelerate

## Compatibility patches (applied to the cloned repo)

In [ ]:
%%writefile /content/patch_confit.py
"""Minimal compatibility patches for running the official ConFit v2 code on one GPU with current libraries."""
import re, sys
from pathlib import Path
root = Path(sys.argv[1])
p = root / "src/model/confit.py"; s = p.read_text()
for name in ("job", "resume"):
    old = f"self.all_gather(batched_all_coarse_{name},sync_grads=True).flatten(0,1).contiguous()"
    new = f"self._gather_flat(batched_all_coarse_{name})"
    if new in s:
        continue            # already patched
    assert old in s, old
    s = s.replace(old, new)
if "_gather_flat" not in s.split("class ConFitModel", 1)[1].split("def __init__", 1)[0]:
    s = s.replace("class ConFitModel(BaseModel):\n", "class ConFitModel(BaseModel):\n"
                  "    def _gather_flat(self, x):\n"
                  "        # one process: all_gather returns x unchanged (2-D); several: (world, batch, dim)\n"
                  "        g = self.all_gather(x, sync_grads=True)\n"
                  "        return (g.flatten(0, 1) if g.dim() == 3 else g).contiguous()\n\n", 1)
p.write_text(s)
t = root / "runners/trainer/train_confit.py"; s = t.read_text()
s = s.replace('accelerator="gpu",', 'accelerator=os.environ.get("PL_ACCEL", "gpu"),')
t.write_text(s)
print("patched")
u = root / "src/utils/test_embedding_networks.py"; s = u.read_text()
s = s.replace('''        assert self.dataset_type in [
            "AliTianChi",
            "recruiting_data",
        ], f"Invalid dataset type: {self.dataset_type}"''', '''        assert self.dataset_type in [
            "AliTianChi",
            "recruiting_data",
            "recruiting_data_v1",
            "recruiting_data_v2",
        ], f"Invalid dataset type: {self.dataset_type}"''')
s = s.replace('model = model.to("cuda")', 'model = model.to(os.environ.get("CONFIT_DEVICE", "cuda"))')
if "\nimport os" not in s:
    s = "import os\n" + s
u.write_text(s)
print("patched test utilities")
e = root / "src/evaluation/eval.py"; s = e.read_text()
old = "        for k, v in self.test_ranking_data.items():\n            k = str(k)\n"
new = old + "            if len(v.get(\"satisfied\", [])) < 2:   # a ranking needs at least two judged items\n                continue\n"
if new not in s:
    assert old in s
    s = s.replace(old, new, 1)
e.write_text(s)
print("patched evaluator")
h = root / "src/utils/hard_negative_mining.py"; s = h.read_text()
s = s.replace('assert self.dataset_type in ["AliTianChi", "recruiting_data"],',
              'assert self.dataset_type in ["AliTianChi", "recruiting_data", "recruiting_data_v1", "recruiting_data_v2"],')
s = s.replace('model = model.to("cuda")', 'model = model.to(os.environ.get("CONFIT_DEVICE", "cuda"))')
h.write_text(s)
m = root / "src/model/confit.py"; s = m.read_text()
# the repo's fallback for Jina v2 points at a placeholder folder; use its own local Jina implementation instead
s = s.replace('JinaBertModel.from_pretrained("path_to_cache_folder_jinav2")', 'JinaBertModel.from_pretrained(args.pretrained_encoder)')
m.write_text(s)
print("patched mining and Jina fallback")

In [ ]:
%%writefile /content/ConFit-v2/confit_scores.py
"""Score every (test job, pool candidate) pair with a trained ConFit v2 checkpoint, using the official
model and dataset code, and write job_id,cv_id,score for score_on_panel.py.

Run from the ConFit-v2 repo root with PYTHONPATH set to it:
    python confit_scores.py --run model_checkpoints/confit_v2_djinni --pairs /content/pairs --out confit_scores.csv
"""
import argparse
import csv
import json
import os
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from src.config.dataset import DATASET_CONFIG
from src.model.confit import ConFitModel, ConFitModelArguments
from src.preprocess.dataset import RJPairSimplifiedDataset, rj_pair_collate_fn


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", required=True, help="the --save_path of the training run")
    ap.add_argument("--ckpt", default=None, help="checkpoint file (default: the best one recorded in the run folder)")
    ap.add_argument("--pairs", required=True)
    ap.add_argument("--out", default="confit_scores.csv")
    ap.add_argument("--dataset-type", default="recruiting_data_v2")
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--jobs-csv", default=None, help="jd_no,text_job file used in training (for example with HyRe text)")
    a = ap.parse_args()
    dev = os.environ.get("CONFIT_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

    run = Path(a.run)
    margs = ConFitModelArguments(**json.loads((run / "model_args.json").read_text()))
    ckpt = a.ckpt
    if ckpt is None:
        cands = sorted(run.glob("epoch=*.ckpt"), key=lambda p: p.stat().st_mtime)
        ckpt = str(cands[-1] if cands else run / "last.ckpt")
    state = torch.load(ckpt, map_location="cpu", weights_only=False)["state_dict"]
    model = ConFitModel(margs)
    model.load_state_dict(state)
    model = model.float().to(dev).eval()
    print("loaded", ckpt)

    pdir = Path(a.pairs)
    pool = [json.loads(l) for l in (pdir / "docs" / "pool.jsonl").open(encoding="utf-8")]
    jobs = {j["id"]: j for j in map(json.loads, (pdir / "docs" / "jobs.jsonl").open(encoding="utf-8"))}
    test_jobs = sorted({r["job_id"] for r in csv.DictReader((pdir / "final_labels.csv").open(encoding="utf-8"))
                        if r["split"] == "test"})
    resumes = [{"user_id": c["id"], "text_resume": c["text"]} for c in pool]
    if a.jobs_csv:
        jt = {r["jd_no"]: r["text_job"] for r in csv.DictReader(open(a.jobs_csv, encoding="utf-8"))}
        jds = [{"jd_no": j, "text_job": jt.get(j, jobs[j]["text"])} for j in test_jobs]
    else:
        jds = [{"jd_no": j, "text_job": jobs[j]["text"]} for j in test_jobs]
    n = max(len(resumes), len(jds))            # pair every id with something so each gets embedded once
    pairs = [{"user_id": resumes[i % len(resumes)]["user_id"], "jd_no": jds[i % len(jds)]["jd_no"], "satisfied": 0}
             for i in range(n)]

    cfg = DATASET_CONFIG[a.dataset_type]
    tok = AutoTokenizer.from_pretrained(margs.pretrained_encoder)
    ds = RJPairSimplifiedDataset(
        tok, max_seq_length_per_key=cfg["max_seq_len_per_feature"], resume_key_names=cfg["resume_key_names"],
        job_key_names=cfg["job_key_names"], tokenizer_args={"padding": "max_length", "return_tensors": "pt", "truncation": True},
        all_resume_dict=resumes, all_job_dict=jds, label_pairs=pairs, resume_taxon_token=cfg["resume_taxon_token"],
        job_taxon_token=cfg["job_taxon_token"], query_prefix="", encode_all=margs.encode_all)
    dl = DataLoader(ds, batch_size=a.batch_size, shuffle=False, collate_fn=rj_pair_collate_fn, num_workers=2)
    R, J = {}, {}
    for bi, batch in enumerate(dl):
        with torch.no_grad():
            out = model.predict_step(batch, bi)
        for off in range(out.batched_resume_representation.shape[0]):
            p = pairs[bi * a.batch_size + off]
            R[p["user_id"]] = out.batched_resume_representation[off].float().cpu().numpy()
            J[p["jd_no"]] = out.batched_job_representation[off].float().cpu().numpy()
    rid = list(R); Rm = np.stack([R[r] for r in rid])
    with open(a.out, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["job_id", "cv_id", "score"])
        for j in test_jobs:
            s = Rm @ J[j]
            w.writerows(zip([j] * len(rid), rid, s.tolist()))
    print(f"wrote {a.out}: {len(test_jobs)} jobs x {len(rid)} candidates")


if __name__ == "__main__":
    main()

In [ ]:
!python /content/patch_confit.py /content/ConFit-v2

## Settings

In [ ]:
import os, json, time, glob, shutil, subprocess, zipfile
SMOKE = True               # True = quick trial (1 epoch, no RUHM); set False for the real run
REPO = "/content/ConFit-v2"
ENCODER = "jinaai/jina-embeddings-v2-base-en"
USE_HYRE = True            # hypothetical resume for every job (ConFit v2 part 1)
USE_RUHM = True            # runner-up hard-negative mining + second training (ConFit v2 part 2)
LLM = "Qwen/Qwen2.5-7B-Instruct"
EPOCHS, BATCH, SEED = 10, 8, 42
if SMOKE: EPOCHS, USE_RUHM = 1, False
MAX_RESUME_TOKENS, MAX_JOB_TOKENS = 1024, 2048     # job text grows when the hypothetical resume is appended
DATA = f"{REPO}/dataset/recruiting_data_v2_djinni"   # the repo requires the dataset type in the path
os.environ.update(PYTHONPATH=REPO, WANDB_MODE="offline", CUDA_VISIBLE_DEVICES="0", TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD="1",
                  TOKENIZERS_PARALLELISM="false")
LOG, T = {}, {}
cfg = f"{REPO}/src/config/proprietary_v2.py"
s = open(cfg).read()
s = s.replace('"text_resume": 2000', f'"text_resume": {MAX_RESUME_TOKENS}').replace('"text_job":3200', f'"text_job": {MAX_JOB_TOKENS}')
open(cfg, "w").write(s)
print("max lengths set:", MAX_RESUME_TOKENS, MAX_JOB_TOKENS)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUN_DIR = "/content/drive/MyDrive/confit_run/" + ("smoke" if SMOKE else "full")   # separate, so the real run never resumes from trial checkpoints
os.makedirs(RUN_DIR, exist_ok=True)
print("run folder:", RUN_DIR)

## Data: upload pairs.zip

In [ ]:
from google.colab import files
DRIVE_ZIP = "/content/drive/MyDrive/confit_run/pairs.zip"
if os.path.exists(DRIVE_ZIP):
    name = DRIVE_ZIP
    print("pairs.zip source: Google Drive,", DRIVE_ZIP)
else:
    up = files.upload()                      # choose pairs.zip
    name = next(iter(up))
    print("pairs.zip source: uploaded file,", name)
shutil.rmtree("/content/pairs", ignore_errors=True)
with zipfile.ZipFile(name) as z:
    z.extractall("/content/pairs_unzipped")
root = next(os.path.dirname(p) for p in glob.glob("/content/pairs_unzipped/**/final_labels.csv", recursive=True))
shutil.move(root, "/content/pairs")
os.makedirs(DATA, exist_ok=True)
for f in glob.glob("/content/pairs/confit/*"):
    shutil.copy(f, DATA)
print(sorted(os.listdir(DATA)))
import pandas as pd
for f in ("train_labeled_data.jsonl", "valid_classification_data.jsonl", "test_classification_data.jsonl"):
    d = pd.read_json(f"{DATA}/{f}", lines=True); print(f, len(d), "pairs,", int(d.satisfied.sum()), "positive")

## 1. HyRe: a hypothetical resume for every job (Qwen2.5-7B-Instruct)

In [ ]:
JOB_CSV = f"{DATA}/all_jd.csv"
HYRE_SAVED = (f"{RUN_DIR}/all_jd_hyre.csv", f"{RUN_DIR}/hyre_generated.json")
if USE_HYRE and all(os.path.exists(f) for f in HYRE_SAVED):
    shutil.copy(HYRE_SAVED[0], f"{DATA}/all_jd_hyre.csv")
    shutil.copy(HYRE_SAVED[1], "/content/hyre_generated.json")
    JOB_CSV = f"{DATA}/all_jd_hyre.csv"
    print("HyRe restored from Drive:", RUN_DIR)
elif USE_HYRE:
    import torch, random, re
    from transformers import AutoTokenizer, AutoModelForCausalLM
    t0 = time.time()
    jd = pd.read_csv(f"{DATA}/all_jd.csv"); rs = pd.read_csv(f"{DATA}/all_resume.csv")
    tr = pd.read_json(f"{DATA}/train_labeled_data.jsonl", lines=True)
    pos = tr[tr.satisfied == 1].sample(frac=1, random_state=SEED).head(5)      # template pairs, as in the paper
    jtext, rtext = dict(zip(jd.jd_no, jd.text_job)), dict(zip(rs.user_id, rs.text_resume))
    templates = [{"job": jtext[r.jd_no][:2500], "resume": rtext[r.user_id][:2500]} for r in pos.itertuples()]
    tok = AutoTokenizer.from_pretrained(LLM); tok.padding_side = "left"
    llm = AutoModelForCausalLM.from_pretrained(LLM, torch_dtype=torch.bfloat16, device_map="cuda")
    rng = random.Random(SEED)
    def prompt(job):   # the official HyRe prompt (src/utils/convert_by_llm.py)
        t = rng.choice(templates)
        p = "\n".join(["Here is a template pair of matching resume and job:", "[The start of the example job]", t["job"],
                        "[The end of the example job]", "[The start of the example resume]", t["resume"],
                        "[The end of the example resume]"
                        "You are a helpful assistant. Following the above example pair of job and resume, construct an ideal resume for the target job shown below. You should strictly follow the format of the given pairs, make sure the resume you give perfectly matches the target job, and directly return your answer in plain texts.",
                        "[The start of the target job]", job[:3000], "[The end of the target job]"])
        return tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
    gen = {}
    ids = list(jd.jd_no)
    for i in range(0, len(ids), 16):
        chunk = ids[i:i + 16]
        enc = tok([prompt(jtext[j]) for j in chunk], return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = llm.generate(**enc, max_new_tokens=512, do_sample=False)
        for j, o in zip(chunk, out):
            text = tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True)
            for pat in (r"\[The start of [^\]]*\]\n?", r"\n?\[The end of [^\]]*\]?"):
                text = re.sub(pat, "", text)
            gen[str(j)] = text.strip()
        print(f"{min(i + 16, len(ids))}/{len(ids)} jobs", flush=True)
    jd["text_job"] = [jtext[j] + "\n[Example resume]\n" + gen[str(j)] for j in jd.jd_no]   # concat=True, as in merge_json.py
    JOB_CSV = f"{DATA}/all_jd_hyre.csv"; jd.to_csv(JOB_CSV, index=False)
    json.dump(gen, open("/content/hyre_generated.json", "w"), indent=1)
    del llm; torch.cuda.empty_cache()
    T["hyre_min"] = round((time.time() - t0) / 60, 1)
    print("HyRe done in", T["hyre_min"], "min; example:\n", jd.text_job.iloc[0][-800:])
    shutil.copy(JOB_CSV, RUN_DIR); shutil.copy("/content/hyre_generated.json", RUN_DIR)
    print("HyRe outputs copied to", RUN_DIR)

## 2. Train ConFit v2 (stage 1)

In [ ]:
def common(job_csv):
    return ["--resume_data_path", f"{DATA}/all_resume.csv", "--job_data_path", job_csv,
            "--num_resume_features", "1", "--num_job_features", "1", "--num_suit_encoder_layers", "0", "--num_encoder_layers", "0",
            "--encode_all", "True", "--embedding_method", "mean_pool", "--finegrained_loss", "noop",
            "--pretrained_encoder", ENCODER, "--do_normalize", "true", "--temperature", "0.05", "--do_both_rj_hard_neg", "true"]

def train(save, extra):
    cmd = ["python", "runners/trainer/train_confit.py", "--save_path", save,
           "--train_label_path", f"{DATA}/train_labeled_data.jsonl", "--valid_label_path", f"{DATA}/valid_classification_data.jsonl",
           "--classification_data_path", f"{DATA}/test_classification_data.jsonl",
           "--rank_resume_data_path", f"{DATA}/rank_resume.json", "--rank_job_data_path", f"{DATA}/rank_job.json",
           "--dataset_type", "recruiting_data_v2", "--train_batch_size", str(BATCH), "--val_batch_size", str(BATCH),
           "--gradient_accumulation_steps", "1", "--weight_decay", "1e-2", "--precision", "bf16-mixed", "--strategy", "auto",
           "--max_epochs", str(EPOCHS), "--seed", str(SEED)] + common(JOB_CSV) + extra
    t0 = time.time()
    with open(f"/content/{os.path.basename(save)}.log", "w") as log:
        p = subprocess.run(cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT)
    T[os.path.basename(save) + "_min"] = round((time.time() - t0) / 60, 1)
    tail = open(f"/content/{os.path.basename(save)}.log").read().splitlines()
    print("\n".join(l for l in tail[-60:] if "it/s]" not in l and "s/it]" not in l))
    assert p.returncode == 0, f"training failed; see /content/{os.path.basename(save)}.log"

def resume_stage(save, n):
    """If RUN_DIR/<stage>/last.ckpt exists, copy that folder back into the repo and skip training."""
    d = f"{RUN_DIR}/{os.path.basename(save)}"
    if not os.path.exists(f"{d}/last.ckpt"):
        return False
    shutil.rmtree(f"{REPO}/{save}", ignore_errors=True)
    shutil.copytree(d, f"{REPO}/{save}")
    if os.path.exists(f"{d}/{os.path.basename(save)}.log"):
        shutil.copy(f"{d}/{os.path.basename(save)}.log", "/content/")
    print(f"resumed stage {n} from Drive")
    return True

def save_stage(save):
    """Copy model_args.json, last.ckpt, the newest epoch=*.ckpt, *.json reports and the log to RUN_DIR/<stage>/."""
    src, d = f"{REPO}/{save}", f"{RUN_DIR}/{os.path.basename(save)}"
    os.makedirs(d, exist_ok=True)
    epochs = sorted(glob.glob(f"{src}/epoch=*.ckpt"), key=os.path.getmtime)
    keep = [f"{src}/model_args.json", f"{src}/last.ckpt", *epochs[-1:], *glob.glob(f"{src}/*.json"),
            f"/content/{os.path.basename(save)}.log"]
    size = 0
    for f in dict.fromkeys(keep):
        if os.path.exists(f):
            shutil.copy2(f, d); size += os.path.getsize(f)
    print(f"copied {size / 1e9:.2f} GB to {d}")

STAGE1 = "model_checkpoints/confit_v2_djinni_stage1"
if not resume_stage(STAGE1, 1):
    train(STAGE1, ["--num_hard_negatives", "0"])
    save_stage(STAGE1)
FINAL = STAGE1

## 3. RUHM: mine runner-up hard negatives, then train again (stage 2)

In [ ]:
if USE_RUHM:
    n_jobs = pd.read_json(f"{DATA}/train_labeled_data.jsonl", lines=True).jd_no.nunique()
    # the paper's band is ranks 3%..4%; with a few hundred jobs keep at least ranks 5..10
    lower, upper = min(0.5, max(0.04, 10 / n_jobs)), min(0.25, max(0.03, 5 / n_jobs))
    ckpt = f"{REPO}/{STAGE1}/last.ckpt"
    cmd = ["python", "src/utils/hard_negative_mining.py", "--model_path", ckpt, "--file_path", "hard_negatives",
           "--file_name", "djinni_hn.json", "--classification_data_path", f"{DATA}/train_labeled_data.jsonl",
           "--rank_resume_data_path", f"{DATA}/valid_rank_resume.json", "--rank_job_data_path", f"{DATA}/valid_rank_job.json",
           "--dataset_type", "recruiting_data_v2", "--batch_size", "16", "--seed", str(SEED),
           "--lower_bound", str(lower), "--upper_bound", str(upper)] + common(JOB_CSV)
    p = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    print(p.stdout[-1500:], p.stderr[-3000:] if p.returncode else "")
    assert p.returncode == 0, "mining failed"
    hn = json.load(open(f"{REPO}/hard_negatives/djinni_hn.json"))
    LOG["hard_negatives"] = {"band": [upper, lower], "resumes": len(hn["r_to_j"]), "jobs": len(hn["j_to_r"])}
    print(LOG["hard_negatives"])
    STAGE2 = "model_checkpoints/confit_v2_djinni_stage2"
    if not resume_stage(STAGE2, 2):
        train(STAGE2, ["--num_hard_negatives", "2", "--hard_negative_path", f"{REPO}/hard_negatives/djinni_hn.json"])
        save_stage(STAGE2)
    FINAL = STAGE2

## 4. Score the test jobs against the whole candidate pool

In [ ]:
p = subprocess.run(["python", "confit_scores.py", "--run", FINAL, "--pairs", "/content/pairs", "--out", "/content/confit_scores.csv",
                    "--jobs-csv", JOB_CSV], cwd=REPO, capture_output=True, text=True)
print(p.stdout[-800:], p.stderr[-2000:] if p.returncode else "")
assert p.returncode == 0

## 5. Manifest and download

In [ ]:
import importlib.metadata as md_, torch
vers = {k: md_.version(k) for k in ("torch", "pytorch_lightning", "transformers", "deepspeed", "faiss-cpu")}
man = {"repo": "github.com/jasonyux/ConFit-v2", "commit": subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
       "encoder": ENCODER, "hyre": USE_HYRE, "hyre_llm": LLM if USE_HYRE else None, "ruhm": USE_RUHM, "epochs": EPOCHS,
       "batch": BATCH, "seed": SEED, "max_tokens": {"resume": MAX_RESUME_TOKENS, "job": MAX_JOB_TOKENS},
       "final_model": FINAL, "gpu": torch.cuda.get_device_name(0), "packages": vers, "minutes": T, **LOG,
       "smoke": SMOKE, "labelled_pairs_note": "192 unresolved pairs dropped (no label)",
       "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
json.dump(man, open("/content/confit_manifest.json", "w"), indent=2)
with zipfile.ZipFile("/content/confit_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["/content/confit_scores.csv", "/content/confit_manifest.json", "/content/hyre_generated.json",
              *glob.glob("/content/*.log"), f"{REPO}/{FINAL}/model_args.json",
              f"{REPO}/{FINAL}/classification_score_report.json"]:
        if os.path.exists(f):
            z.write(f, os.path.basename(f))
print(json.dumps(man, indent=1))
for f in ("/content/confit_results.zip", "/content/confit_manifest.json"):
    shutil.copy(f, RUN_DIR)
print("results copied to", RUN_DIR)
files.download("/content/confit_results.zip")